# Delta Lake com Apache Spark

Este notebook demonstra como usar Delta Lake com PySpark para criar uma tabela transacional local.

- Criação e escrita de tabela Delta
- Leitura e consulta de dados
- Atualização de registros
- Time travel

In [7]:
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip


builder = (
    SparkSession.builder
    .appName("DeltaLakeStudy")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

delta_path = os.path.abspath("tmp/delta_table")
os.makedirs(os.path.dirname(delta_path), exist_ok=True)

data = [(1, "Alice", 1200), (2, "Bob", 1500), (3, "Carol", 1600)]
df = spark.createDataFrame(data, ["id", "name", "salary"])

df.write.format("delta").mode("overwrite").save(delta_path)
print(f"Tabela Delta escrita em: {delta_path}")

Tabela Delta escrita em: /home/gabriel/GitHub/datalakehouse-study/notebooks/tmp/delta_table


In [8]:
spark.read.format("delta").load(delta_path).show()

spark.sql("DROP TABLE IF EXISTS delta_demo")
spark.sql(f"CREATE TABLE delta_demo USING DELTA LOCATION '{delta_path}'")
spark.sql("SELECT * FROM delta_demo").show()

+---+-----+------+
| id| name|salary|
+---+-----+------+
|  1|Alice|  1200|
|  3|Carol|  1600|
|  2|  Bob|  1500|
+---+-----+------+

+---+-----+------+
| id| name|salary|
+---+-----+------+
|  1|Alice|  1200|
|  3|Carol|  1600|
|  2|  Bob|  1500|
+---+-----+------+



In [9]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, delta_path)
delta_table.update("id = 1", {"salary": "salary + 1400"})
print("Registro atualizado para id = 1")

spark.read.format("delta").load(delta_path).show()

Registro atualizado para id = 1
+---+-----+------+
| id| name|salary|
+---+-----+------+
|  1|Alice|  2600|
|  3|Carol|  1600|
|  2|  Bob|  1500|
+---+-----+------+



In [ ]:

new_data = [(4, "David", 1800), (5, "Eve", 2000)]
new_df = spark.createDataFrame(new_data, ["id", "name", "salary"])
new_df.write.format("delta").mode("append").save(delta_path)
print("Novos registros inseridos")

spark.read.format("delta").load(delta_path).show()

In [10]:
print("Versão 0 - antes da atualização")
spark.read.format("delta").option("versionAsOf", 0).load(delta_path).show()

Versão 0 - antes da atualização
+---+-----+------+
| id| name|salary|
+---+-----+------+
|  1|Alice|  1200|
|  3|Carol|  1600|
|  2|  Bob|  1500|
+---+-----+------+



In [ ]:

delta_table.delete("id = 2")
print("Registro com id = 2 deletado")

spark.read.format("delta").load(delta_path).show()